# 1.2 Data for the Survival Analysis

**For Class 5 (3.4 survival_analysis):** pull what we need to study **time from a diabetes diagnosis to chronic kidney disease (CKD)**.

For everyone in our population **with diabetes** we pull:

| Table | What | Saved as |
|---|---|---|
| Diagnoses | every diabetes and CKD diagnosis, with its date | `survival_conditions.csv` |
| Visits | each person's **last** visit date (for people who never get CKD) | `last_visits.csv` |

Run **1.1 build_cohorts** first only if you haven't set up `aou_setup.R` yet; this notebook doesn't need its files.

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data or counts of 1 to 20. Clear outputs before you share or commit this notebook.

In [ ]:
source("aou_setup.R")  # settings, packages, and helper functions

## Query 1: Diabetes and CKD diagnoses

Each row is one diagnosis, flagged:

- `is_diabetes`: a diabetes mellitus code (concept 201820 or any more specific type)
- `is_ckd`: a chronic kidney disease code (our CKD concept set and its more specific codes)

A code like *"CKD stage 3 due to type 2 diabetes"* counts as **both**.

In [ ]:
ckd_codes <- "37017104, 37017813, 37018761, 43531559, 43531562, 43531566, 43531577, 43531578, 43531653, 443597, 443601, 443611, 443612, 443614, 44782429, 44782689, 44782690, 44782691, 44782692, 45763854, 45763855, 45769901, 45769903, 45771075, 45773576, 45773688, 46270353, 46271022, 46273636"

conditions_sql <- str_glue("{cohort_sql()},
diabetes_codes AS (
  SELECT DISTINCT descendant_concept_id AS concept_id
  FROM `{CURRENT_DATASET}.concept_ancestor`
  WHERE ancestor_concept_id = 201820
),
ckd_codes AS (
  SELECT DISTINCT descendant_concept_id AS concept_id
  FROM `{CURRENT_DATASET}.concept_ancestor`
  WHERE ancestor_concept_id IN ({ckd_codes})
)
SELECT
  co.person_id,
  co.condition_start_date,
  cc.concept_name AS condition_name,
  dc.concept_id IS NOT NULL AS is_diabetes,
  kc.concept_id IS NOT NULL AS is_ckd
FROM `{CURRENT_DATASET}.condition_occurrence` co
JOIN cohort c ON co.person_id = c.person_id
LEFT JOIN diabetes_codes dc ON co.condition_concept_id = dc.concept_id
LEFT JOIN ckd_codes      kc ON co.condition_concept_id = kc.concept_id
LEFT JOIN `{CURRENT_DATASET}.concept` cc ON co.condition_concept_id = cc.concept_id
WHERE c.diabetes = 'Yes'
  AND (dc.concept_id IS NOT NULL OR kc.concept_id IS NOT NULL)
")

survival_conditions <- run_query(conditions_sql)
glimpse(survival_conditions)

In [ ]:
save_data(survival_conditions, "survival_conditions.csv")

## Query 2: Last visit date

People who never develop CKD are **censored** at their last visit: we know they were CKD-free until then. SQL's `MAX()` finds each person's latest visit. We count outpatient, inpatient, emergency, and telehealth visits.

In [ ]:
visits_sql <- str_glue("{cohort_sql()}
SELECT
  v.person_id,
  MAX(COALESCE(v.visit_end_date, v.visit_start_date)) AS last_visit_date
FROM `{CURRENT_DATASET}.visit_occurrence` v
JOIN cohort c ON v.person_id = c.person_id
WHERE c.diabetes = 'Yes'
  AND v.visit_concept_id IN (9201, 9202, 9203, 581477, 5083)
GROUP BY v.person_id
")

last_visits <- run_query(visits_sql)
glimpse(last_visits)

In [ ]:
save_data(last_visits, "last_visits.csv")
list.files(DATA_DIR)